# Notebook 04 — Customer Segmentation

**Project:** CX Intelligence — NPS & Complaint Severity Analysis  
**Author:** Nicolás Zuleta Sierra

## Objectives
1. Build a feature matrix from severity and NLP-derived features
2. Select optimal k with Elbow method + Silhouette score
3. Train K-Means and assign customers to segments
4. Profile each segment statistically
5. Name segments with CX-actionable labels
6. Visualize clusters in 2D (PCA) with Plotly
7. Provide CX action recommendations per segment

**Features used (updated from notebook 02 pivot):**
- `severity_label` — predicted complaint severity (LOW / MEDIUM / HIGH)
- `severity_proba_high` — probability of HIGH severity (continuous risk signal)
- `dominant_topic` — LDA topic index (from notebook 03)
- `nps_score` — simulated NPS score
- `product_encoded` — banking product ordinal encoding
- `complaint_length` — word count (frustration depth proxy)

**Replaced:** `finbert_label`, `finbert_score` (sentiment models achieved ~33% accuracy = random chance)

**Input:** `data/processed/features_nlp.csv` (with severity columns from Notebook 02, topic columns from Notebook 03)  
**Output:** `models/kmeans_model.joblib` + cluster columns in features_nlp.csv

## 0. Imports & Configuration

In [4]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.segmentation import (
    assign_clusters,
    build_cluster_profiles,
    build_feature_matrix,
    find_optimal_k,
    name_clusters,
    save_kmeans_model,
    train_kmeans,
)

plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 80)

FEATURES_PATH = ROOT / "data" / "processed" / "features_nlp.csv"
KMEANS_MODEL_PATH = ROOT / "models" / "kmeans_model.joblib"

print(f"Features exist: {FEATURES_PATH.exists()}")

Features exist: True


## 1. Load Features Dataset

In [5]:
df = pd.read_csv(FEATURES_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Verify required columns (updated for severity pivot)
required = ["severity_encoded", "severity_proba_high", "dominant_topic", "nps_score", "product"]
missing = [c for c in required if c not in df.columns]
if missing:
    print(f"WARNING — Missing columns: {missing}")
    print("Run Notebooks 01, 02, and 03 first.")
else:
    print("All required columns present.")

# Show severity distribution as sanity check
if "severity_label" in df.columns:
    print("\nSeverity distribution:")
    print(df["severity_label"].value_counts())

df.head(2)

Shape: (35226, 19)
Columns: ['complaint_id', 'consumer_complaint_narrative', 'text_clean', 'lemmatized_text', 'product', 'submitted_via', 'date_received', 'company_response_to_consumer', 'timely_response', 'severity', 'severity_encoded', 'severity_label', 'severity_proba_low', 'severity_proba_medium', 'severity_proba_high', 'dominant_topic', 'topic_name', 'topic_probability', 'lda_text']
WARNING — Missing columns: ['nps_score']
Run Notebooks 01, 02, and 03 first.

Severity distribution:
severity_label
MEDIUM    27105
LOW        7216
HIGH        905
Name: count, dtype: int64


,complaint_id,consumer_complaint_narrative,text_clean,lemmatized_text,product,submitted_via,date_received,company_response_to_consumer,timely_response,severity,severity_encoded,severity_label,severity_proba_low,severity_proba_medium,severity_proba_high,dominant_topic,topic_name,topic_probability,lda_text
0,1290492,Due to inconsistencies in the amount owed that I was told by M & T Bank and ...,due to inconsistencies in the amount owed that i was told by m t bank and th...,inconsistency owe tell bank report credit reporting agency advise write good...,Personal loan,Web,03/19/2015,Closed with explanation,Yes,LOW,0,MEDIUM,0.301,0.6904,0.0086,3.0,Credit Reporting & Disputes,0.870976,inconsistency owe reporting agency advise write good order address issue neg...
1,1290524,"In XX/XX/XXXX my wages that I earned at my job decreased by almost half, by ...",in my wages that i earned at my job decreased by almost half by i knew i was...,wage earn job decrease half know trouble home loan begin contact wfb home lo...,Mortgage,Web,03/19/2015,Closed with explanation,Yes,MEDIUM,1,MEDIUM,0.294,0.6974,0.0086,0.0,Incorrect Charges & Unauthorized Debits,0.568080,wage earn job decrease half trouble home begin wfb home assitance option ear...


In [6]:
# Merge NPS scores from banking_complaints.csv (produced by notebook 01)
BANKING_PATH = ROOT / "data" / "processed" / "banking_complaints.csv"
if BANKING_PATH.exists() and "nps_score" not in df.columns:
    nps_cols = pd.read_csv(
        BANKING_PATH,
        usecols=["complaint_id", "nps_score", "nps_segment"],
        low_memory=False,
    )
    df = df.merge(nps_cols, on="complaint_id", how="left")
    print(f"NPS merged. nps_score null: {df["nps_score"].isna().sum():,} rows")
else:
    print("NPS columns already present")
print(f"Shape after merge: {df.shape}")

NPS merged. nps_score null: 0 rows
Shape after merge: (35226, 21)


## 2. Build Feature Matrix

Features selected for clustering (updated after pivot from sentiment to severity):

| Feature | Type | Source |
|---------|------|--------|
| `severity_encoded` | Ordinal (0/1/2) | Notebook 02 — XGBoost prediction |
| `severity_proba_high` | Continuous (0–1) | Notebook 02 — XGBoost P(HIGH) |
| `dominant_topic` | Ordinal | Notebook 03 — LDA topic index |
| `nps_score` | Continuous (0–10) | Notebook 01 — simulated NPS |
| `product_encoded` | Ordinal | Notebook 02 — product severity weight |
| `complaint_length` | Continuous | Notebook 02 — word count |

All features are standardized (z-score) before clustering.

In [7]:
# Drop rows with any missing core feature (severity columns are required)
required_for_clustering = ["severity_encoded", "severity_proba_high", "dominant_topic", "nps_score"]
df_cluster = df.dropna(subset=required_for_clustering).copy()
print(f"Rows available for clustering: {len(df_cluster):,} (dropped {len(df) - len(df_cluster):,} with missing values)")

X, le = build_feature_matrix(df_cluster)
print(f"Feature matrix shape: {X.shape}")

Rows available for clustering: 35,220 (dropped 6 with missing values)
Feature matrix shape: (35220, 5)


## 3. Elbow Method

In [8]:
print("Computing Elbow + Silhouette for k=2 to 8…")
k_results = find_optimal_k(X, k_range=range(2, 9))

elbow_df = pd.DataFrame(
    [{"k": k, "inertia": v} for k, v in k_results["elbow"].items()]
)

fig = px.line(
    elbow_df, x="k", y="inertia",
    markers=True,
    title="Elbow Method — Within-Cluster Sum of Squares (WCSS)",
    labels={"k": "Number of Clusters (k)", "inertia": "WCSS (Inertia)"},
    color_discrete_sequence=["#1e40af"],
)
fig.show()

Computing Elbow + Silhouette for k=2 to 8…


## 4. Silhouette Score

In [9]:
silhouette_df = pd.DataFrame(
    [{"k": k, "silhouette": v} for k, v in k_results["silhouette"].items()]
)

fig = px.bar(
    silhouette_df, x="k", y="silhouette",
    color="silhouette",
    color_continuous_scale=["#fee2e2", "#22c55e"],
    title="Silhouette Score by k",
    labels={"k": "Number of Clusters (k)", "silhouette": "Silhouette Score"},
    text="silhouette",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(coloraxis_showscale=False)
fig.show()

optimal_k = silhouette_df.loc[silhouette_df["silhouette"].idxmax(), "k"]
print(f"Optimal k (max silhouette): {optimal_k}")

Optimal k (max silhouette): 7


## 5. Train K-Means

In [10]:
N_CLUSTERS = int(optimal_k)  # Override here if needed
print(f"Training K-Means with k={N_CLUSTERS}…")

kmeans = train_kmeans(X, n_clusters=N_CLUSTERS)
df_cluster = assign_clusters(df_cluster, kmeans, X)

print("Cluster distribution:")
print(df_cluster["cluster"].value_counts().sort_index())

Training K-Means with k=7…
Cluster distribution:
cluster
0    6200
1    3073
2    7471
3    4908
4     905
5    5287
6    7376
Name: count, dtype: int64


## 6. Cluster Profiles

In [11]:
profiles = build_cluster_profiles(df_cluster)
print("=== CLUSTER PROFILES ===")
print(profiles.to_string(index=False))

=== CLUSTER PROFILES ===
 cluster  n_complaints  avg_nps dominant_severity  avg_severity_proba_high                 top_product  pct_detractors  pct_high_severity
       4           905     5.11              HIGH                   0.8977                    Mortgage            68.6              100.0
       0          6200     6.46            MEDIUM                   0.0088 Credit card or prepaid card            43.4                0.0
       2          7471     6.46            MEDIUM                   0.0095                    Mortgage            43.5                0.0
       5          5287     6.46            MEDIUM                   0.0099                    Mortgage            43.9                0.0
       1          3073     8.39            MEDIUM                   0.0082                    Mortgage             0.0                0.0
       3          4908     8.48            MEDIUM                   0.0073                    Mortgage             0.1                0.0
       6 

## 7. CX Segment Naming

Clusters are ranked by average NPS and assigned a CX-actionable name.  
Names reflect both the data profile and the recommended intervention.

In [12]:
cx_name_map = name_clusters(profiles)

print("\n=== CX SEGMENT NAMES ===")
for cluster_id, name in sorted(cx_name_map.items()):
    row = profiles[profiles["cluster"] == cluster_id].iloc[0]
    print(f"Cluster {cluster_id} → {name}")
    print(f"  NPS: {row['avg_nps']} | Sentiment: {row['dominant_severity']} | "
          f"Detractors: {row['pct_detractors']}% | n={row['n_complaints']:,}")

df_cluster["cluster_name"] = df_cluster["cluster"].map(cx_name_map)


=== CX SEGMENT NAMES ===
Cluster 0 → Silent Dissatisfied
  NPS: 6.46 | Sentiment: MEDIUM | Detractors: 43.4% | n=6,200
Cluster 1 → Active Promoters
  NPS: 8.39 | Sentiment: MEDIUM | Detractors: 0.0% | n=3,073
Cluster 2 → Neutral Observers
  NPS: 6.46 | Sentiment: MEDIUM | Detractors: 43.5% | n=7,471
Cluster 3 → Promoter
  NPS: 8.48 | Sentiment: MEDIUM | Detractors: 0.1% | n=4,908
Cluster 4 → Critical Risk
  NPS: 5.11 | Sentiment: HIGH | Detractors: 68.6% | n=905
Cluster 5 → Promoter Candidates
  NPS: 6.46 | Sentiment: MEDIUM | Detractors: 43.9% | n=5,287
Cluster 6 → Promoter
  NPS: 8.74 | Sentiment: LOW | Detractors: 0.0% | n=7,376


## 8. PCA Scatter Plot — 2D Cluster Visualization

In [13]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

df_cluster["pca_1"] = X_pca[:, 0]
df_cluster["pca_2"] = X_pca[:, 1]

variance_explained = pca.explained_variance_ratio_
print(f"Variance explained by PCA: {variance_explained[0]:.1%} + {variance_explained[1]:.1%} = {sum(variance_explained):.1%}")

hover_cols = [c for c in ["nps_score", "severity_label", "product"] if c in df_cluster.columns]

fig = px.scatter(
    df_cluster.sample(min(5000, len(df_cluster)), random_state=42),  # sample for speed
    x="pca_1", y="pca_2",
    color="cluster_name",
    opacity=0.5,
    title=f"Customer Segments — PCA Projection (k={N_CLUSTERS})",
    labels={
        "pca_1": f"PC1 ({variance_explained[0]:.1%} variance)",
        "pca_2": f"PC2 ({variance_explained[1]:.1%} variance)",
    },
    hover_data=hover_cols if hover_cols else None,
)
fig.update_layout(height=550, legend_title="CX Segment")
fig.show()

Variance explained by PCA: 41.6% + 20.9% = 62.4%


## 9. Product Distribution by Segment

In [14]:
if "product" in df_cluster.columns:
    prod_dist = (
        df_cluster.groupby(["cluster_name", "product"])
        .size()
        .reset_index(name="count")
    )
    fig = px.bar(
        prod_dist, x="cluster_name", y="count", color="product",
        barmode="stack",
        title="Product Distribution by CX Segment",
        labels={"count": "Complaints", "cluster_name": "Segment"},
    )
    fig.update_layout(xaxis_tickangle=-20, height=450)
    fig.show()

## 10. CX Action Recommendations

In [15]:
cx_recommendations = {
    "Critical Risk": (
        "Immediate personal outreach within 48h. Escalate to specialized retention team. "
        "Assign a dedicated case manager. High churn probability."
    ),
    "Silent Dissatisfied": (
        "Proactive NPS survey recommended. Neutral language masks real frustration. "
        "Targeted recovery campaign. Do not ignore — this segment churns quietly."
    ),
    "Neutral Observers": (
        "Monitor for sentiment drift. Opportunity for proactive engagement. "
        "Educational content and proactive service updates can move this group toward Promoters."
    ),
    "Promoter Candidates": (
        "Activate for referral programs. Request public reviews on Google/Trustpilot. "
        "Cross-sell premium products. Low intervention cost, high ROI."
    ),
    "Active Promoters": (
        "Brand ambassador program. Referral incentives. Minimal service intervention needed. "
        "Maintain satisfaction — avoid disrupting what is working."
    ),
}

print("=== CX ACTION RECOMMENDATIONS ===")
for segment, rec in cx_recommendations.items():
    if segment in cx_name_map.values():
        print(f"\n[{segment}]")
        print(f"  {rec}")

=== CX ACTION RECOMMENDATIONS ===

[Critical Risk]
  Immediate personal outreach within 48h. Escalate to specialized retention team. Assign a dedicated case manager. High churn probability.

[Silent Dissatisfied]
  Proactive NPS survey recommended. Neutral language masks real frustration. Targeted recovery campaign. Do not ignore — this segment churns quietly.

[Neutral Observers]
  Monitor for sentiment drift. Opportunity for proactive engagement. Educational content and proactive service updates can move this group toward Promoters.

[Promoter Candidates]
  Activate for referral programs. Request public reviews on Google/Trustpilot. Cross-sell premium products. Low intervention cost, high ROI.

[Active Promoters]
  Brand ambassador program. Referral incentives. Minimal service intervention needed. Maintain satisfaction — avoid disrupting what is working.


## 11. Save Model and Updated Dataset

In [16]:
# Save K-Means model
save_kmeans_model(kmeans, KMEANS_MODEL_PATH)

# Merge cluster results back into features_nlp.csv
cluster_cols = ["complaint_id", "cluster", "cluster_name", "pca_1", "pca_2"]
cluster_cols = [c for c in cluster_cols if c in df_cluster.columns]

df_full = pd.read_csv(FEATURES_PATH, low_memory=False)
df_cluster_export = df_cluster[cluster_cols].copy()

# Drop existing cluster columns if re-running
for col in ["cluster", "cluster_name", "pca_1", "pca_2"]:
    if col in df_full.columns:
        df_full = df_full.drop(columns=[col])

df_full = df_full.merge(df_cluster_export, on="complaint_id", how="left")
df_full.to_csv(FEATURES_PATH, index=False)
print(f"Updated features_nlp.csv with cluster assignments: {len(df_full):,} rows")

# Show sample with new severity + cluster columns
display_cols = [c for c in ["complaint_id", "nps_score", "severity_label", "cluster", "cluster_name"] if c in df_full.columns]
df_full[display_cols].head()

K-Means model saved → /Users/nicolaszuleta95/code_nz/cx-intelligence-nps/models/kmeans_model.joblib
Updated features_nlp.csv with cluster assignments: 35,226 rows


,complaint_id,severity_label,cluster,cluster_name
0,1290492,MEDIUM,3.0,Promoter
1,1290524,MEDIUM,5.0,Promoter Candidates
2,1290253,MEDIUM,5.0,Promoter Candidates
3,1292137,MEDIUM,3.0,Promoter
4,1290254,MEDIUM,2.0,Neutral Observers


## 12. Summary

**Optimal k: 7** (max silhouette score) · PCA variance explained: 62.4% (PC1: 41.6%, PC2: 20.9%)

**Segment profiles:**

| Segment | Avg NPS | Dominant Severity | Top Product | % Detractors | % HIGH Severity | n |
|---------|---------|------------------|-------------|--------------|-----------------|---|
| Critical Risk | 5.11 | HIGH | Mortgage | 68.6% | **100.0%** | 905 |
| Silent Dissatisfied | 6.46 | MEDIUM | Credit card | 43.4% | 0.0% | 6,200 |
| Neutral Observers | 6.46 | MEDIUM | Mortgage | 43.5% | 0.0% | 7,471 |
| Promoter Candidates | 6.46 | MEDIUM | Mortgage | 43.9% | 0.0% | 5,287 |
| Active Promoters | 8.39 | MEDIUM | Mortgage | 0.0% | 0.0% | 3,073 |
| Promoter | 8.48 | MEDIUM | Mortgage | 0.1% | 0.0% | 4,908 |
| Promoter | 8.74 | LOW | Credit card | 0.0% | 0.0% | 7,376 |

**Key insights:**

- **Critical Risk (n=905, 2.6%):** Every complaint is HIGH severity. Mortgage-driven, 68.6% Detractors — highest-priority escalation target. Despite being the smallest segment, it concentrates all HIGH-severity cases.

- **The MEDIUM-NPS cluster (6.46 NPS, ~43% Detractors):** Three clusters share an identical average NPS. Severity and topic distinguish them — these customers are dissatisfied but not actively complaining at HIGH severity. Standard NPS monitoring alone misses this heterogeneity.

- **Promoter tiers (8.39–8.74 NPS):** Three clusters with near-zero Detractors and low severity. Differentiated by product: Credit card customers (Cluster 6) show slightly higher NPS and LOW severity dominance.

- **The diagnostic blind spot:** Filtering by NPS alone (threshold < 7) would combine Critical Risk with Silent Dissatisfied — missing that one group has 100% HIGH severity and the other has 0%. Severity + NPS together reveal the structure that NPS alone cannot.

**CX allocation recommendation:**
1. Critical Risk → Dedicated case manager + immediate outreach
2. Silent Dissatisfied / Neutral / Candidates → Proactive recovery campaign, prioritised by severity_proba_high
3. Promoter tiers → Referral program activation, cross-sell